In [24]:
!pip install -q -U --no-cache-dir \
    langchain-core langchain-community langchain-groq \
    langchain-text-splitters langchain-huggingface langgraph \
    faiss-cpu sentence-transformers pypdf "gradio==6.19.0" groq

In [25]:
!pip uninstall -y gradio -q
!pip install -q --no-cache-dir "gradio==6.19.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 110.7 MB/s eta 0:00:00


In [26]:
import os
import re
import json
import getpass
from collections import Counter, defaultdict

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

import gradio as gr

print("Packages imported")
print("Gradio version:", gr.__version__)
if not gr.__version__.startswith("6.19"):
    print("Gradio is not pinned to 6.19.0 — re-run the pip install cell above, "
          "then Runtime → Restart session, then run all cells again from the top.")


Packages imported
Gradio version: 6.19.0


In [27]:
from google.colab import files

uploaded = files.upload()
PDF_PATH = list(uploaded.keys())[0]
print(f"Uploaded: {PDF_PATH}")


Saving Generative_AI_Question_Paper.pdf to Generative_AI_Question_Paper (4).pdf
Uploaded: Generative_AI_Question_Paper (4).pdf


In [28]:
os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your GROQ_API_KEY: ")
print("Groq API key set")

Enter your GROQ_API_KEY: ··········
Groq API key set


In [29]:
GROQ_MODEL = "openai/gpt-oss-120b"

llm = ChatGroq(
    model=GROQ_MODEL,
    temperature=0.2,
    reasoning_effort="low",   # "low" | "medium" (default) | "high" — lower = faster
    groq_api_key=os.environ["GROQ_API_KEY"],
)
print(f"Connected to Groq model: {GROQ_MODEL}")

Connected to Groq model: openai/gpt-oss-120b


In [30]:
response = llm.invoke("In one sentence, what is database normalization?")
print(response.content)

Database normalization is the systematic process of organizing data into tables to eliminate redundancy and ensure logical relationships, thereby improving data integrity and reducing update anomalies.


In [31]:
loader = PyPDFLoader(PDF_PATH)
pages = loader.load()
full_text = "\n".join(p.page_content for p in pages)

print(f"Loaded {len(pages)} page(s), {len(full_text)} characters")

Loaded 2 page(s), 2269 characters


In [32]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80,
    separators=["\n\n", "\n", ".", " "],
)
chunks = splitter.split_documents(pages)
print(f"Created {len(chunks)} chunks")


Created 6 chunks


In [33]:
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print("Embedding model ready")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model ready


In [34]:
vector_db = FAISS.from_documents(chunks, embedding_model)
print("FAISS vector database created")

FAISS vector database created


In [35]:
retriever = vector_db.as_retriever(search_kwargs={"k": 4})
print("Retriever ready")


Retriever ready


In [36]:
results = retriever.invoke("normalization")
for i, r in enumerate(results, 1):
    print(f"--- match {i} ---")
    print(r.page_content[:300])
    print()

--- match 1 ---
9. Define fine-tuning in the context of large language models. (2 marks)
10. What is mode collapse in GANs? (2 marks)
PART B — Answer any FOUR (4 × 5 = 20 Marks)
11. Explain the architecture of a Generative Adversarial Network (GAN) with a diagram. (5 marks)
12. Describe the encoder-decoder structur

--- match 2 ---
marks)
14. Describe the forward and reverse diffusion process in diffusion models. (5 marks)
15. Explain the concept of attention and multi-head attention in Transformer models. (5 marks)
16. Discuss the ethical concerns and biases associated with generative AI models. (5 marks)
PART C — Answer any 

--- match 3 ---
2. What is a Generative Adversarial Network (GAN)? (2 marks)
3. What is prompt engineering? (2 marks)
4. Differentiate between discriminative and generative models. (2 marks)
5. Define a Variational Autoencoder (VAE). (2 marks)
6. What is the self-attention mechanism used in Transformers? (2 marks)


--- match 4 ---
Generative Adversarial Network

In [37]:
def answer_from_paper(query: str) -> str:
    docs = retriever.invoke(query)
    context = "\n\n".join(d.page_content for d in docs)
    prompt = (
        "Answer the question using ONLY the excerpts below from the uploaded question paper. "
        "If the excerpts don't contain the answer, say so.\n\n"
        f"Excerpts:\n{context}\n\nQuestion: {query}"
    )
    return llm.invoke(prompt).content

print("RAG function ready")
print(answer_from_paper("What topics does this question paper cover?"))


RAG function ready
The excerpts show that the paper covers the following topics:

- Generative Adversarial Networks (GANs) – definition, architecture, training process, challenges such as mode collapse, and diagrammatic explanation.  
- Prompt engineering – definition and critical analysis of techniques (zero‑shot, few‑shot, chain‑of‑thought).  
- Discriminative vs. generative models – comparison.  
- Variational Autoencoders (VAEs) – definition and encoder‑decoder structure.  
- Self‑attention mechanism in Transformers – description.  
- Attention and multi‑head attention in Transformer models.  
- Diffusion models – applications, forward and reverse diffusion processes.  
- Ethical concerns and biases in generative AI models.  
- Fine‑tuning of large language models.  
- Normalization techniques (e.g., batch normalization) for stabilising GAN training.


In [38]:
@tool
def question_paper_rag(query: str) -> str:
    """Search the uploaded question paper and answer questions about its content —
    e.g. finding questions on a topic, explaining a question, or summarizing what
    the paper covers. Input should be a natural-language question."""
    return answer_from_paper(query)

print("✅ RAG tool created")

✅ RAG tool created


In [39]:
QUESTION_SPLIT_RE = re.compile(r'(?m)^\s*(?:Q(?:uestion)?\.?\s*)?(\d{1,2})[\.\)]\s+')
MARKS_RE = re.compile(r'\[?\(?\s*(\d{1,3})\s*(?:marks?|m)\s*\)?\]?', re.IGNORECASE)

TOPIC_KEYWORDS = {
    "Normalization": ["normalization", "1nf", "2nf", "3nf", "bcnf", "normal form"],
    "SQL": ["sql", "select", "join", "query", "queries", "database language"],
    "ER Model": ["er model", "entity", "relationship diagram", "er diagram", "cardinality"],
    "Transactions": ["transaction", "acid", "commit", "rollback", "concurrency"],
    "Indexing": ["index", "b-tree", "b+ tree", "hashing"],
    "Keys & Constraints": ["primary key", "foreign key", "candidate key", "constraint"],
    "Algorithms": ["algorithm", "complexity", "big-o", "sorting", "searching"],
    "Data Structures": ["stack", "queue", "linked list", "tree", "graph", "array"],
    "Networking": ["network", "protocol", "tcp", "ip address", "osi"],
    "Operating Systems": ["process", "thread", "scheduling", "deadlock", "memory management"],
    "GANs": ["gan", "generative adversarial", "mode collapse", "discriminator", "generator network"],
    "VAEs": ["vae", "variational autoencoder", "encoder-decoder", "latent space"],
    "Diffusion Models": ["diffusion", "denoising", "forward process", "reverse process"],
    "Transformers & Attention": ["transformer", "attention", "self-attention", "positional encoding"],
    "Prompt Engineering": ["prompt engineering", "zero-shot", "few-shot", "chain-of-thought"],
    "Generative AI Basics": ["generative ai", "generative model", "discriminative model"],
}

def split_into_questions(text: str):
    matches = list(QUESTION_SPLIT_RE.finditer(text))
    questions = []
    for i, m in enumerate(matches):
        start = m.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        q_text = text[start:end].strip()
        if len(q_text) > 5:
            questions.append(q_text)
    if not questions:
        questions = [ln.strip() for ln in text.split("\n") if len(ln.strip()) > 15]
    return questions

def extract_marks(q_text: str):
    m = MARKS_RE.search(q_text)
    return int(m.group(1)) if m else None

def tag_topic(q_text: str) -> str:
    lower = q_text.lower()
    for topic, keywords in TOPIC_KEYWORDS.items():
        if any(kw in lower for kw in keywords):
            return topic
    return "Other / Uncategorized"

def estimate_difficulty(q_text: str, marks) -> str:
    lower = q_text.lower()
    hard_signals = ["design", "prove", "derive", "compare and contrast", "differentiate", "critically"]
    easy_signals = ["define", "list", "what is", "state"]
    if any(s in lower for s in hard_signals) or (marks and marks >= 10):
        return "Hard"
    if any(s in lower for s in easy_signals) or (marks and marks <= 2):
        return "Easy"
    return "Medium"

def normalize_for_dedup(q_text: str) -> str:
    q = re.sub(r'^\s*\d{1,2}[\.\)]\s*', '', q_text)
    q = re.sub(r'\[?\(?\s*\d{1,3}\s*(?:marks?|m)\s*\)?\]?', '', q, flags=re.IGNORECASE)
    q = re.sub(r'\W+', ' ', q).strip().lower()
    return q

def find_repeated_questions(questions):
    seen = defaultdict(list)
    for q in questions:
        key = normalize_for_dedup(q)
        bucket = " ".join(key.split()[:8])
        seen[bucket].append(q)
    return [group for group in seen.values() if len(group) > 1]

def analyze_paper(_input: str = "") -> str:
    """Run full structural analysis of the uploaded question paper and return a
    JSON string with counts, topics, marks distribution, difficulty, repeated
    questions, AND the verbatim question text grouped by marks value — so exact
    question lists never need to be guessed or paraphrased from memory."""
    questions = split_into_questions(full_text)
    marks_list = [extract_marks(q) for q in questions]
    topics = [tag_topic(q) for q in questions]
    difficulties = [estimate_difficulty(q, m) for q, m in zip(questions, marks_list)]
    repeated = find_repeated_questions(questions)

    marks_counter = Counter(m for m in marks_list if m is not None)
    topic_counter = Counter(topics)
    difficulty_counter = Counter(difficulties)

    # Group verbatim question text by marks value — this is what lets the agent
    # answer "give me the 10-mark questions" with the ACTUAL text instead of
    # inventing plausible-sounding but wrong questions.
    questions_by_marks = defaultdict(list)
    for q, m in zip(questions, marks_list):
        if m is not None:
            questions_by_marks[m].append(q.strip())

    report = {
        "total_questions": len(questions),
        "marks_distribution": dict(sorted(marks_counter.items())),
        "questions_by_marks": {str(k): v for k, v in sorted(questions_by_marks.items())},
        "topic_frequency": dict(topic_counter.most_common()),
        "difficulty_breakdown": dict(difficulty_counter),
        "repeated_question_groups": len(repeated),
        "repeated_questions_full": repeated,
        "questions_missing_marks": marks_list.count(None),
    }
    return json.dumps(report, indent=2)

@tool
def exam_analysis_tool(query: str = "") -> str:
    """Use this for ANYTHING involving counting, listing, or structure in the question
    paper: total question count, marks distribution, the EXACT TEXT of questions worth
    a specific mark value, repeated/duplicate questions (verbatim), topic frequency, or
    difficulty breakdown. Always prefer this over guessing — it returns the real
    question text from the paper, grouped by marks and flagged for repeats, so answers
    should quote directly from its output rather than paraphrase from general knowledge.
    Input can be empty; it always analyzes the full uploaded paper.
    """
    return analyze_paper(query)

print("Analysis tool created")
print(analyze_paper(""))

Analysis tool created
{
  "total_questions": 20,
  "marks_distribution": {
    "2": 10,
    "5": 6,
    "10": 4
  },
  "questions_by_marks": {
    "2": [
      "1. Define Generative AI and give two real-world examples. (2 marks)",
      "2. What is a Generative Adversarial Network (GAN)? (2 marks)",
      "3. What is prompt engineering? (2 marks)",
      "4. Differentiate between discriminative and generative models. (2 marks)",
      "5. Define a Variational Autoencoder (VAE). (2 marks)",
      "6. What is the self-attention mechanism used in Transformers? (2 marks)",
      "7. List two applications of diffusion models. (2 marks)",
      "8. What is prompt engineering? (2 marks)",
      "9. Define fine-tuning in the context of large language models. (2 marks)",
      "10. What is mode collapse in GANs? (2 marks)\nPART B \u2014 Answer any FOUR (4 \u00d7 5 = 20 Marks)"
    ],
    "5": [
      "11. Explain the architecture of a Generative Adversarial Network (GAN) with a diagram. (5 mark

In [40]:
tools = [question_paper_rag, exam_analysis_tool]

agent = create_react_agent(llm, tools=tools)

result = agent.invoke({
    "messages": [{"role": "user", "content": "How many questions are present in the paper, and what is the marks distribution?"}]
})
print(result["messages"][-1].content)


/tmp/ipykernel_8394/504015421.py:3: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools=tools)


**Total number of questions:** **20**

**Marks distribution**

| Marks per question | Number of questions | Example questions |
|--------------------|---------------------|-------------------|
| **2 marks** | 10 | 1. Define Generative AI… (2 marks) <br>2. What is a Generative Adversarial Network (GAN)? (2 marks) … |
| **5 marks** | 6 | 11. Explain the architecture of a GAN with a diagram. (5 marks) <br>12. Describe the encoder‑decoder structure of a VAE. (5 marks) … |
| **10 marks** | 4 | 17. Discuss in detail the architecture, training process, and challenges of GANs. (10 marks) <br>18. Compare and contrast VAEs, GANs, and Diffusion Models… (10 marks) … |

So the paper contains **20 questions** split into **10 short‑answer (2‑mark) items, 6 medium‑length (5‑mark) items, and 4 long‑answer (10‑mark) items**.


In [41]:
memory = MemorySaver()
print("Memory created")

Memory created


In [42]:
agent_with_memory = create_react_agent(llm, tools=tools, checkpointer=memory)
thread_config = {"configurable": {"thread_id": "student-session-1"}}

def ask_agent(question: str) -> str:
    result = agent_with_memory.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config=thread_config,
    )
    return result["messages"][-1].content

# quick smoke test
print(ask_agent("What are the important topics in this question paper?"))


/tmp/ipykernel_8394/1773618846.py:1: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_with_memory = create_react_agent(llm, tools=tools, checkpointer=memory)


**Key topics covered in the paper**

| Topic | How many questions (by mark‑value) |
|-------|------------------------------------|
| **Generative AI basics** (definition, examples) | 2‑mark Q1 |
| **Generative Adversarial Networks (GANs)** | Q2 (2 marks), Q11 (5 marks), Q13 (5 marks), Q17 (10 marks) |
| **Prompt engineering** | Q3 & Q8 (duplicate 2‑mark), Q9 (2 marks), Q20 (10 marks) |
| **Variational Auto‑encoders (VAEs)** | Q5 (2 marks), Q12 (5 marks), Q18 (10 marks) |
| **Transformers & self‑attention** | Q6 (2 marks), Q15 (5 marks), Q19 (10 marks) |
| **Diffusion models** | Q7 (2 marks), Q14 (5 marks), Q18 (10 marks) |
| **Discriminative vs. generative models** | Q4 (2 marks) |
| **Fine‑tuning of large language models** | Q9 (2 marks) |
| **Mode collapse in GANs** | Q10 (2 marks), Q17 (10 marks) |
| **Normalization (e.g., batch‑norm) for GAN stability** | Q13 (5 marks) |
| **Ethical concerns & bias in generative AI** | Q16 (5 marks) |
| **General comparison of major generative‑mode

In [ ]:
from google.colab import files
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

vector_db = None
retriever = None
full_text = ""
CURRENT_THREAD_ID = "thread-initial"
chat_log = []

def process_pdf(path):
    global full_text, vector_db, retriever, CURRENT_THREAD_ID
    import uuid
    loader = PyPDFLoader(path)
    pages = loader.load()
    full_text = "\n".join(p.page_content for p in pages)
    new_chunks = splitter.split_documents(pages)
    vector_db = FAISS.from_documents(new_chunks, embedding_model)
    retriever = vector_db.as_retriever(search_kwargs={"k": 4})
    CURRENT_THREAD_ID = f"thread-{uuid.uuid4().hex[:8]}"
    print(f"✅ Indexed \"{path}\" — {len(new_chunks)} chunks. Ask away!")

def ask_agent(question: str) -> str:
    if retriever is None:
        return "⚠️ Please upload a PDF first (run the upload cell below)."
    config = {"configurable": {"thread_id": CURRENT_THREAD_ID}}
    result = agent_with_memory.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config=config,
    )
    return result["messages"][-1].content

# --- Upload a PDF ---
print("📄 Upload your question paper:")
uploaded = files.upload()
process_pdf(list(uploaded.keys())[0])

# --- Simple chat loop ---
print("\n💬 Type your question and press Enter. Type 'quit' to stop.\n")
while True:
    q = input("You: ")
    if q.strip().lower() in ("quit", "exit"):
        print("👋 Done.")
        break
    answer = ask_agent(q)
    print(f"\nAgent: {answer}\n")

📄 Upload your question paper:


Saving Generative_AI_Question_Paper.pdf to Generative_AI_Question_Paper (2).pdf
✅ Indexed "Generative_AI_Question_Paper (2).pdf" — 6 chunks. Ask away!

💬 Type your question and press Enter. Type 'quit' to stop.

You: quit
👋 Done.


In [43]:
import uuid

vector_db = None
retriever = None
full_text = ""
CURRENT_THREAD_ID = "thread-initial"

def process_pdf(file_obj):
    """Re-runs the load -> chunk -> embed -> FAISS -> retriever pipeline on a newly
    uploaded PDF, and starts a fresh memory thread so old answers don't bleed into
    the new paper."""
    global full_text, vector_db, retriever, CURRENT_THREAD_ID

    if file_obj is None:
        return "⚠️ Please upload a PDF first.", []

    pdf_path = file_obj if isinstance(file_obj, str) else file_obj.name

    loader = PyPDFLoader(pdf_path)
    pages = loader.load()
    full_text = "\n".join(p.page_content for p in pages)

    new_chunks = splitter.split_documents(pages)
    vector_db = FAISS.from_documents(new_chunks, embedding_model)
    retriever = vector_db.as_retriever(search_kwargs={"k": 4})

    CURRENT_THREAD_ID = f"thread-{uuid.uuid4().hex[:8]}"  # fresh memory for this paper

    status = f"✅ Indexed \"{os.path.basename(pdf_path)}\" — {len(new_chunks)} chunks. Ask away!"
    return status, []


def ask_agent(question: str) -> str:
    if retriever is None:
        return "⚠️ Please upload a question paper PDF first."
    config = {"configurable": {"thread_id": CURRENT_THREAD_ID}}
    result = agent_with_memory.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config=config,
    )
    return result["messages"][-1].content


with gr.Blocks(title="🎓 Exam Intelligence Agent") as demo:
    gr.Markdown(
        "## 🎓 AI-Based Autonomous Examination Intelligence Agent\n"
        "Upload a question paper PDF, then ask things like:\n"
        "- What are the important topics in this question paper?\n"
        "- Which questions are repeated?\n"
        "- What is the marks distribution?\n"
        "- Create a revision plan from this question paper."
    )

    with gr.Row():
        pdf_input = gr.File(label="Upload question paper (PDF)", file_types=[".pdf"])
        status_box = gr.Textbox(label="Status", interactive=False, value="No PDF uploaded yet.")

    chatbot = gr.Chatbot(label="Ask about the paper", height=420)  # no `type` kwarg on 6.19.0
    msg = gr.Textbox(label="Your question", placeholder="e.g. What is the marks distribution?")
    clear_btn = gr.Button("Clear chat")

    pdf_input.change(fn=process_pdf, inputs=pdf_input, outputs=[status_box, chatbot])

    def respond(message, chat_history):
        if chat_history is None:
            chat_history = []
        reply = ask_agent(message)
        chat_history = chat_history + [
            {"role": "user", "content": message},
            {"role": "assistant", "content": reply},
        ]
        return "", chat_history

    msg.submit(respond, [msg, chatbot], [msg, chatbot])
    clear_btn.click(lambda: [], None, chatbot)

try:
    demo.close()
except Exception:
    pass

demo.launch(debug=True, share=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://d65691c7014b9911f7.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


KeyboardInterrupt: 

In [46]:
!pip install -q pyngrok

In [ ]:
from pyngrok import ngrok
import getpass, time

NGROK_TOKEN = getpass.getpass("Paste your ngrok authtoken: ")
ngrok.set_auth_token(NGROK_TOKEN)
ngrok.kill()

try:
    gr.close_all()
except Exception:
    pass

# Start Gradio FIRST, non-blocking, so it's actually listening before
# we open a tunnel to it (this ordering is what causes the 504).
demo.launch(debug=False, share=False, server_port=7860, prevent_thread_lock=True)

time.sleep(5)  # give the server time to fully bind before tunneling to it

public_url = ngrok.connect(7860)
print(f"🌐 Open this in your browser: {public_url}")

# Keep this cell alive so the server (and tunnel) keep running
while True:
    time.sleep(60)


Paste your ngrok authtoken: ··········
Closing server running on port: 7860
Closing server running on port: 7861
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

🌐 Open this in your browser: NgrokTunnel: "https://ferret-dropout-glitch.ngrok-free.dev" -> "http://localhost:7860"


/usr/local/lib/python3.13/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.13/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


In [ ]:
import subprocess
print(subprocess.run(["pip", "show", "gradio"], capture_output=True, text=True).stdout)

import gradio
print("Imported version:", gradio.__version__)
print("Imported from:", gradio.__file__)

Name: gradio
Version: 6.19.0
Summary: Python library for easily interacting with trained machine learning models
Home-page: https://github.com/gradio-app/gradio
Author: 
Author-email: Abubakar Abid <gradio-team@huggingface.co>, Ali Abid <gradio-team@huggingface.co>, Ali Abdalla <gradio-team@huggingface.co>, Dawood Khan <gradio-team@huggingface.co>, Ahsen Khaliq <gradio-team@huggingface.co>, Pete Allen <gradio-team@huggingface.co>, Ömer Faruk Özdemir <gradio-team@huggingface.co>, Freddy A Boulton <gradio-team@huggingface.co>, Hannah Blair <gradio-team@huggingface.co>
License: 
Location: /usr/local/lib/python3.13/dist-packages
Requires: anyio, audioop-lts, brotli, fastapi, gradio-client, groovy, hf-gradio, httpx, huggingface-hub, jinja2, markupsafe, numpy, orjson, packaging, pandas, pillow, pydantic, pydub, python-multipart, pytz, pyyaml, safehttpx, semantic-version, starlette, tomlkit, typer, typing-extensions, uvicorn
Required-by: 

Imported version: 6.19.0
Imported from: /usr/local/li